# Utility Placement Optimisation

**Learning outcome:** Apply utility placement optimisation through the public `PinchProblem` or `PinchWorkspace` workflow.

**Level:** Advanced  
**Execution profile:** `base`  
**Expected runtime:** under 2 minutes  
**Optional extras:** plot

The lifecycle is explicit: prepare the study, run the named method, then inspect cached results. Observation cells do not launch analysis.

## Study question and data

**Study question:** Where should four isothermal hot and cold utility levels be placed at Process and Site hierarchy levels to minimize thermodynamic cost?

The sample data is packaged with OpenPinch, so the notebook runs without path setup. Read the named inputs and assumptions before substituting plant data.

## Step 1: Prepare the placement study

Run this cell once, then inspect its named outputs. Arguments on the method call apply to this analysis; stored configuration is only the fallback when an argument is omitted.

In [9]:
import pandas as pd

from OpenPinch import PinchWorkspace

workspace = PinchWorkspace(
    source="chocolate_factory.json", project_name="Site"
)
problem = workspace.use_case("baseline")
baseline_input = problem.to_problem_json()

def retarget_comparison(evidence, target):
    period = evidence.best.period_results[0]
    rows = []
    for side, levels, utilities in (
        ("hot", period.hot_levels, target.hot_utilities),
        ("cold", period.cold_levels, target.cold_utilities),
    ):
        retargeted = {utility.name: float(utility.heat_flow.value) for utility in utilities}
        for level in levels:
            name = level.template_key.name
            rows.append({
                "side": side,
                "utility": name,
                "kind": level.kind.value,
                "supply_degC": level.supply_temperature.value,
                "target_degC": level.target_temperature.value,
                "optimizer_duty_kW": level.allocated_duty.value,
                "retargeted_duty_kW": retargeted.get(name, 0.0),
                "difference_kW": retargeted.get(name, 0.0) - level.allocated_duty.value,
                "fallback": level.is_fallback,
            })
    return pd.DataFrame(rows)

search_options = {
    "iteration_limit": 100,
    "evaluation_limit": 100,
    "candidate_limit": 20,
    "run_count": 10,
}

## Step 2: Optimize a Process Zone and build its standard GCC

Run this cell once, then inspect its named outputs. Arguments on the method call apply to this analysis; stored configuration is only the fallback when an argument is omitted.

In [10]:
process_case = problem.target.utility_placement(
    isothermal=4,
    sensible=0,
    zone="Almond",
    period_ids=("0",),
    options=search_options,
)
process_evidence = process_case.utility_placement_result
process_objective = process_evidence.best.aggregate_objective
process_fallback_penalty = process_evidence.best.fallback_penalty
process_utilities = process_case.to_problem_json()["utilities"]
registered_process = workspace.add(
    process_case,
    name="optimized_process_utilities",
    activate=False,
)
process_target = process_case.results.targets[-1]
process_retarget_comparison = retarget_comparison(
    process_evidence, process_target
)
process_summary = process_case.summary_frame()
process_gcc = process_case.plot.grand_composite_curve(
    zone_name="Almond"
)

## Step 3: Optimize the Site and build its standard Total Site Profile

Run this cell once, then inspect its named outputs. Arguments on the method call apply to this analysis; stored configuration is only the fallback when an argument is omitted.

In [11]:
site_case = problem.target.utility_placement(
    isothermal=4,
    sensible=0,
    period_ids=("0",),
    options=search_options,
)
site_evidence = site_case.utility_placement_result
site_objective = site_evidence.best.aggregate_objective
site_utilities = site_case.to_problem_json()["utilities"]
registered_site = workspace.add(
    site_case,
    name="optimized_site_utilities",
    activate=False,
)
baseline_unchanged = workspace.use_case("baseline").to_problem_json() == baseline_input
site_target = site_case.results.targets[-1]
site_retarget_comparison = retarget_comparison(site_evidence, site_target)
site_summary = site_case.summary_frame()
site_tsp = site_case.plot.total_site_profiles()

## Review the result

Review both optimized cases exactly like normal cases: compare the Process utilities on the standard GCC with the Site utilities on the standard Total Site Profile. The Process Utility GCC must not cross the Process GCC. The comparison tables place optimizer-evidence and cached allocation duties side by side; a zero difference shows exact replay.

In [12]:
from IPython.display import display

display(process_objective)
display(process_fallback_penalty)
display(process_retarget_comparison)
display(process_summary)
display(process_gcc)
display(site_objective)
display(site_retarget_comparison)
display({"baseline_unchanged": baseline_unchanged})
display(site_summary)
display(site_tsp)

QuantityValue(value=0.04776560405295249, unit='kW/K')

QuantityValue(value=0.0, unit='dimensionless')

,side,utility,kind,supply_degC,target_degC,optimizer_duty_kW,retargeted_duty_kW,difference_kW,fallback
0,hot,hot_iso_1,isothermal,173.001408,172.991408,60.745547,60.745547,0.000000e+00,False
1,hot,hot_iso_2,isothermal,125.454389,125.444389,49.440844,49.440844,0.000000e+00,False
2,hot,hot_iso_3,isothermal,81.238633,81.228633,29.030009,29.030009,2.842171e-14,False
3,hot,hot_iso_4,isothermal,21.977472,21.967472,0.000000,0.000000,0.000000e+00,False
4,cold,cold_iso_1,isothermal,172.991408,173.001408,0.000000,0.000000,0.000000e+00,False
5,cold,cold_iso_2,isothermal,125.444389,125.454389,0.000000,0.000000,0.000000e+00,False
6,cold,cold_iso_3,isothermal,81.228633,81.238633,0.000000,0.000000,0.000000e+00,False
7,cold,cold_iso_4,isothermal,21.967472,21.977472,12.136300,12.136300,0.000000e+00,False


,Scope,Zone Type,Integration Type,Target Method,Period ID,Hot Utility Target,Cold Utility Target,Heat Recovery,Hot Pinch,Cold Pinch,Hot Utilities,Cold Utilities
0,Site/Almond,Process Zone,Process,Heat Exchange,0,139.22 kW,12.14 kW,51.35 kW,33.00 degC,33.00 degC,"hot_iso_1: 60.75 kW, hot_iso_2: 49.44 kW, hot_...","cold_iso_1: 0.00 kW, cold_iso_2: 0.00 kW, cold..."


QuantityValue(value=0.45414397788230154, unit='kW/K')

,side,utility,kind,supply_degC,target_degC,optimizer_duty_kW,retargeted_duty_kW,difference_kW,fallback
0,hot,hot_iso_1,isothermal,173.046997,173.036997,726.927059,726.927059,0.000000e+00,False
1,hot,hot_iso_2,isothermal,57.186191,57.176191,555.870971,555.870971,0.000000e+00,False
2,hot,hot_iso_3,isothermal,27.985332,27.975332,0.000000,0.000000,0.000000e+00,False
3,hot,hot_iso_4,isothermal,21.270115,21.260115,0.000000,0.000000,0.000000e+00,False
4,cold,cold_iso_1,isothermal,173.036997,173.046997,0.000000,0.000000,0.000000e+00,False
5,cold,cold_iso_2,isothermal,57.176191,57.186191,0.000000,0.000000,0.000000e+00,False
6,cold,cold_iso_3,isothermal,27.975332,27.985332,1750.300554,1750.300554,4.547474e-13,False
7,cold,cold_iso_4,isothermal,21.260115,21.270115,1851.653419,1851.653419,0.000000e+00,False


{'baseline_unchanged': True}

,Scope,Zone Type,Integration Type,Target Method,Period ID,Hot Utility Target,Cold Utility Target,Heat Recovery,Hot Pinch,Cold Pinch,Hot Utilities,Cold Utilities
0,Site,Site,Process,Heat Exchange,0,1082.54 kW,3401.69 kW,440.47 kW,33.00 degC,33.00 degC,"hot_iso_1: 691.12 kW, hot_iso_2: 391.42 kW, ho...","cold_iso_1: 0.00 kW, cold_iso_2: 0.00 kW, cold..."
1,Site,Site,Utility,Heat Exchange,0,1282.80 kW,3601.95 kW,240.21 kW,57.18 degC,27.99 degC,"hot_iso_1: 726.93 kW, hot_iso_2: 555.87 kW, ho...","cold_iso_1: 0.00 kW, cold_iso_2: 0.00 kW, cold..."


## Interpret the result

Compare the Process result against its direct GCC and the Site result against its Total Site Profile. Candidate duties come from those exact ordinary target workflows; they are not independent optimizer decisions, and a requested level may be unused. Inspect physical entropy generation from the balanced composite curves using the signed Q / T limit for the isothermal intervals; CP * ln(T_out / T_in) in kelvin applies when adapting the workflow to sensible utilities. Confirm that final allocation reproduces every selected duty and that the Utility GCC does not cross the Process GCC.

## Adapt this template

Replace the sample with validated plant data, apply defensible temperature bounds, and increase the optimizer limits before making an engineering decision.

Keep the workflow explicit: prepare input, call one named engineering method, inspect cached results, then export.